<h1><img src="../../../icons/tk_full_logo.svg" width="80" /> AI Lab Education</h1>

# Compare Base vs Fine-Tuned: Tool Use Evaluation

This notebook evaluates whether fine-tuning improved the model's ability to use tools.

## The Experiment

In notebook 03, we observed that **base gpt-oss-20b ignores tools** - it generates text from memory instead of calling the tools we provide.

In notebook 04, we **fine-tuned on synthetic tool-use data** to teach the model:
1. WHEN to call tools (user asks to search → call search tool)
2. HOW to format tool calls (correct JSON structure)
3. WHAT to do with results (incorporate tool output naturally)

**This notebook measures the improvement.**

## Evaluation Methodology

```
┌─────────────────────────────────────────────────────────────────────────┐
│                    Tool-Use Evaluation Pipeline                         │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│  TEST PROMPTS                                                           │
│  ┌──────────────────────────────────────────────────────────────────┐  │
│  │ "Find papers about transformer architectures"                    │  │
│  │ "Log my analysis to MLflow"                                      │  │
│  │ "Get details on the FouRA paper"                                 │  │
│  └──────────────────────────────────────────────────────────────────┘  │
│                              │                                          │
│              ┌───────────────┴───────────────┐                         │
│              ▼                               ▼                         │
│   ┌──────────────────┐           ┌──────────────────┐                  │
│   │   Base Model     │           │  Fine-Tuned      │                  │
│   │  gpt-oss-20b     │           │  gpt-oss-tool    │                  │
│   └────────┬─────────┘           └────────┬─────────┘                  │
│            │                              │                            │
│            ▼                              ▼                            │
│   ┌──────────────────┐           ┌──────────────────┐                  │
│   │ Usually ignores  │           │ Calls tools      │                  │
│   │ tools, generates │           │ correctly with   │                  │
│   │ from memory      │           │ proper JSON      │                  │
│   └──────────────────┘           └──────────────────┘                  │
│                                                                         │
│  METRICS                                                                │
│  • Tool call rate: % of prompts that trigger tool calls                │
│  • Correct tool: Did it pick the right tool for the task?              │
│  • Valid JSON: Are tool arguments properly formatted?                   │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

## Prerequisites

- Complete notebook 04 (fine-tuning) and deploy the fine-tuned model
- Both models registered in LiteLLM: `gpt-oss-20b` and `gpt-oss-tool-use`

## Setup

Connect to LiteLLM and verify both models are available.

In [ ]:
import os
import json
from openai import OpenAI
import pandas as pd
from IPython.display import display, Markdown

# Validate we're running in the correct JupyterHub image
# This notebook requires agent-dev for CrewAI
from check_jupyter_flavor import check_flavor
check_flavor('agent-dev')

# Platform services
LITELLM_ENDPOINT = os.environ.get('LITELLM_API_BASE') or os.environ.get('LITELLM_ENDPOINT')
LITELLM_KEY = os.environ.get('LITELLM_API_KEY') or os.environ.get('LITELLM_MASTER_KEY')

# If LITELLM_KEY not set, try to load from kubectl secret (for development)
if not LITELLM_KEY:
    import subprocess
    try:
        result = subprocess.run(
            ['kubectl', 'get', 'secret', '-n', 'litellm', 'litellm-secrets', 
             '-o', 'jsonpath={.data.LITELLM_MASTER_KEY}'],
            capture_output=True, text=True, timeout=5
        )
        if result.returncode == 0 and result.stdout:
            import base64
            LITELLM_KEY = base64.b64decode(result.stdout).decode()
    except Exception:
        pass

client = OpenAI(base_url=LITELLM_ENDPOINT, api_key=LITELLM_KEY)

# Model names
BASE_MODEL = "openai/gpt-oss-20b"
FINETUNED_MODEL = "openai/gpt-oss-tool-use"  # Update if deployed with different name

print(f"LiteLLM: {LITELLM_ENDPOINT}")
print(f"Base model: {BASE_MODEL}")
print(f"Fine-tuned model: {FINETUNED_MODEL}")
print(f"\nNote: If fine-tuned model not yet deployed, this notebook will show base model behavior only.")

---
## Define Test Tools

These are the same tools we use in notebook 03. We'll pass them to both models and see which one uses them.

In [ ]:
# OpenAI function calling format - the tools definition
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search_papers",
            "description": "Search the Qdrant vector database for papers matching a semantic query. Use this when you need to find papers about a specific topic or technique.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "A natural language description of what you're looking for (e.g., 'memory efficient LoRA techniques')"
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_paper_details",
            "description": "Get the full content of a specific paper by title. Use this when you need detailed information about a paper you've identified.",
            "parameters": {
                "type": "object",
                "properties": {
                    "paper_title_fragment": {
                        "type": "string",
                        "description": "Part of the paper title to search for (e.g., 'FouRA' or 'LoRA-FA')"
                    }
                },
                "required": ["paper_title_fragment"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "log_to_mlflow",
            "description": "Create an MLflow experiment run to log research findings. Use this to record paper analysis results for future reference.",
            "parameters": {
                "type": "object",
                "properties": {
                    "experiment_name": {
                        "type": "string",
                        "description": "Name for the MLflow experiment"
                    },
                    "run_name": {
                        "type": "string",
                        "description": "Name for this specific run"
                    },
                    "findings_summary": {
                        "type": "string",
                        "description": "A summary of the key findings to log"
                    }
                },
                "required": ["experiment_name", "run_name", "findings_summary"]
            }
        }
    }
]

print(f"Defined {len(TOOLS)} tools for evaluation:")
for tool in TOOLS:
    print(f"  - {tool['function']['name']}: {tool['function']['description'][:60]}...")

---
## Define Test Prompts

Each prompt is designed to trigger a specific tool. We'll measure:
1. **Tool call rate**: Did the model call ANY tool?
2. **Correct tool**: Did it call the RIGHT tool?
3. **Valid arguments**: Are the arguments properly formatted?

In [ ]:
# Test cases: (prompt, expected_tool_name)
TEST_CASES = [
    # Search papers tests
    ("Find papers about transformer architectures", "search_papers"),
    ("Search for research on memory-efficient fine-tuning", "search_papers"),
    ("What papers exist about LoRA variants?", "search_papers"),
    ("Look up research on attention mechanisms", "search_papers"),
    
    # Get paper details tests
    ("Get me the full details of the FouRA paper", "get_paper_details"),
    ("Show me the content of the LoRA-FA paper", "get_paper_details"),
    ("I need the abstract of the Flat-LoRA paper", "get_paper_details"),
    
    # Log to MLflow tests
    ("Log my analysis of LoRA techniques to MLflow", "log_to_mlflow"),
    ("Create an MLflow experiment for this research session", "log_to_mlflow"),
    ("Save my findings about transformer efficiency to the experiment tracker", "log_to_mlflow"),
]

print(f"Defined {len(TEST_CASES)} test cases:")
print(f"  - {len([t for t in TEST_CASES if t[1] == 'search_papers'])} for search_papers")
print(f"  - {len([t for t in TEST_CASES if t[1] == 'get_paper_details'])} for get_paper_details")
print(f"  - {len([t for t in TEST_CASES if t[1] == 'log_to_mlflow'])} for log_to_mlflow")

---
## Evaluation Function

Test a model on all prompts and collect metrics.

In [ ]:
def evaluate_model(model_name: str, test_cases: list, tools: list, verbose: bool = True) -> dict:
    """
    Evaluate a model's tool-use capability.
    
    Returns:
        dict with metrics: tool_call_rate, correct_tool_rate, valid_json_rate, details
    """
    results = []
    
    for prompt, expected_tool in test_cases:
        result = {
            "prompt": prompt,
            "expected_tool": expected_tool,
            "called_tool": False,
            "tool_name": None,
            "correct_tool": False,
            "valid_json": False,
            "tool_args": None,
            "raw_response": None,
            "error": None
        }
        
        try:
            response = client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": prompt}],
                tools=tools,
                tool_choice="auto",
                max_tokens=500,
                temperature=0.3  # Lower temperature for more deterministic behavior
            )
            
            message = response.choices[0].message
            result["raw_response"] = message.content
            
            # Check if model made a tool call
            if message.tool_calls and len(message.tool_calls) > 0:
                tool_call = message.tool_calls[0]
                result["called_tool"] = True
                result["tool_name"] = tool_call.function.name
                result["correct_tool"] = (tool_call.function.name == expected_tool)
                
                # Try to parse arguments as JSON
                try:
                    args = json.loads(tool_call.function.arguments)
                    result["valid_json"] = True
                    result["tool_args"] = args
                except json.JSONDecodeError:
                    result["valid_json"] = False
                    result["tool_args"] = tool_call.function.arguments
        
        except Exception as e:
            result["error"] = str(e)
        
        results.append(result)
        
        if verbose:
            status = "TOOL" if result["called_tool"] else "NO TOOL"
            correct = "correct" if result["correct_tool"] else "wrong" if result["called_tool"] else "-"
            print(f"  [{status}] {prompt[:50]}... → {result['tool_name'] or 'N/A'} ({correct})")
    
    # Calculate metrics
    total = len(results)
    called = sum(1 for r in results if r["called_tool"])
    correct = sum(1 for r in results if r["correct_tool"])
    valid = sum(1 for r in results if r["valid_json"])
    
    return {
        "model": model_name,
        "total_tests": total,
        "tool_calls": called,
        "correct_tools": correct,
        "valid_json": valid,
        "tool_call_rate": called / total * 100,
        "correct_tool_rate": correct / total * 100,
        "valid_json_rate": valid / total * 100 if called > 0 else 0,
        "details": results
    }

print("Evaluation function defined.")

---
## Evaluate Base Model

Test the original gpt-oss-20b without fine-tuning.

In [ ]:
print(f"Evaluating BASE model: {BASE_MODEL}")
print("=" * 70)

base_results = evaluate_model(BASE_MODEL, TEST_CASES, TOOLS)

print("\n" + "=" * 70)
print(f"BASE MODEL RESULTS:")
print(f"  Tool call rate:    {base_results['tool_call_rate']:.1f}% ({base_results['tool_calls']}/{base_results['total_tests']})")
print(f"  Correct tool rate: {base_results['correct_tool_rate']:.1f}% ({base_results['correct_tools']}/{base_results['total_tests']})")
print(f"  Valid JSON rate:   {base_results['valid_json_rate']:.1f}% ({base_results['valid_json']}/{base_results['tool_calls']} tool calls)")

---
## Evaluate Fine-Tuned Model

Test the model fine-tuned on synthetic tool-use data.

**Note**: If the fine-tuned model is not yet deployed, this cell will fail. In that case, you can:
1. Complete notebook 04 and deploy the model
2. Update `FINETUNED_MODEL` to match your deployment name
3. Re-run this cell

In [ ]:
print(f"Evaluating FINE-TUNED model: {FINETUNED_MODEL}")
print("=" * 70)

try:
    finetuned_results = evaluate_model(FINETUNED_MODEL, TEST_CASES, TOOLS)
    
    print("\n" + "=" * 70)
    print(f"FINE-TUNED MODEL RESULTS:")
    print(f"  Tool call rate:    {finetuned_results['tool_call_rate']:.1f}% ({finetuned_results['tool_calls']}/{finetuned_results['total_tests']})")
    print(f"  Correct tool rate: {finetuned_results['correct_tool_rate']:.1f}% ({finetuned_results['correct_tools']}/{finetuned_results['total_tests']})")
    print(f"  Valid JSON rate:   {finetuned_results['valid_json_rate']:.1f}% ({finetuned_results['valid_json']}/{finetuned_results['tool_calls']} tool calls)")
    
    finetuned_available = True

except Exception as e:
    print(f"\nFine-tuned model not available: {e}")
    print("\nTo complete this comparison:")
    print("  1. Run notebook 04 to fine-tune and deploy the model")
    print("  2. Update FINETUNED_MODEL variable if using a different name")
    print("  3. Re-run this cell")
    finetuned_available = False
    finetuned_results = None

---
## Side-by-Side Comparison

Compare the two models on all metrics.

In [ ]:
# Create comparison table
comparison_data = {
    "Metric": [
        "Tool Call Rate",
        "Correct Tool Rate",
        "Valid JSON Rate"
    ],
    "Base Model": [
        f"{base_results['tool_call_rate']:.1f}%",
        f"{base_results['correct_tool_rate']:.1f}%",
        f"{base_results['valid_json_rate']:.1f}%"
    ]
}

if finetuned_available:
    comparison_data["Fine-Tuned"] = [
        f"{finetuned_results['tool_call_rate']:.1f}%",
        f"{finetuned_results['correct_tool_rate']:.1f}%",
        f"{finetuned_results['valid_json_rate']:.1f}%"
    ]
    comparison_data["Improvement"] = [
        f"+{finetuned_results['tool_call_rate'] - base_results['tool_call_rate']:.1f}%",
        f"+{finetuned_results['correct_tool_rate'] - base_results['correct_tool_rate']:.1f}%",
        f"+{finetuned_results['valid_json_rate'] - base_results['valid_json_rate']:.1f}%"
    ]
else:
    comparison_data["Fine-Tuned"] = ["N/A", "N/A", "N/A"]
    comparison_data["Improvement"] = ["N/A", "N/A", "N/A"]

df = pd.DataFrame(comparison_data)
print("\nCOMPARISON RESULTS:")
print("=" * 70)
display(df)

In [ ]:
# Visualize comparison
import matplotlib.pyplot as plt

metrics = ["Tool Call Rate", "Correct Tool Rate", "Valid JSON Rate"]
base_values = [
    base_results['tool_call_rate'],
    base_results['correct_tool_rate'],
    base_results['valid_json_rate']
]

fig, ax = plt.subplots(figsize=(10, 6))
x = range(len(metrics))
width = 0.35

bars1 = ax.bar([i - width/2 for i in x], base_values, width, label='Base Model', color='#ff6b6b')

if finetuned_available:
    finetuned_values = [
        finetuned_results['tool_call_rate'],
        finetuned_results['correct_tool_rate'],
        finetuned_results['valid_json_rate']
    ]
    bars2 = ax.bar([i + width/2 for i in x], finetuned_values, width, label='Fine-Tuned', color='#4ecdc4')

ax.set_ylabel('Percentage (%)')
ax.set_title('Tool-Use Capability: Base vs Fine-Tuned Model')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend()
ax.set_ylim(0, 100)

# Add value labels on bars
for bar in bars1:
    height = bar.get_height()
    ax.annotate(f'{height:.1f}%',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3),
                textcoords="offset points",
                ha='center', va='bottom')

if finetuned_available:
    for bar in bars2:
        height = bar.get_height()
        ax.annotate(f'{height:.1f}%',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom')

plt.tight_layout()
plt.show()

---
## Detailed Analysis

Look at specific examples where the models differed.

In [ ]:
print("DETAILED COMPARISON BY TEST CASE")
print("=" * 90)

for i, (prompt, expected) in enumerate(TEST_CASES):
    base_detail = base_results['details'][i]
    
    print(f"\n[{i+1}] {prompt}")
    print(f"    Expected: {expected}")
    print(f"    Base:     {base_detail['tool_name'] or 'NO TOOL CALL'} ", end="")
    if base_detail['called_tool']:
        print(f"({'correct' if base_detail['correct_tool'] else 'WRONG'})")
        print(f"              Args: {base_detail['tool_args']}")
    else:
        response_preview = (base_detail['raw_response'] or '')[:100]
        print(f"\n              Response: {response_preview}...")
    
    if finetuned_available:
        ft_detail = finetuned_results['details'][i]
        print(f"    FT:       {ft_detail['tool_name'] or 'NO TOOL CALL'} ", end="")
        if ft_detail['called_tool']:
            print(f"({'correct' if ft_detail['correct_tool'] else 'WRONG'})")
            print(f"              Args: {ft_detail['tool_args']}")
        else:
            response_preview = (ft_detail['raw_response'] or '')[:100]
            print(f"\n              Response: {response_preview}...")

---
## CrewAI Comparison

The raw API tests above show tool-calling capability in isolation. Now let's test both models in the **actual CrewAI multi-agent context** from notebook 03.

This is the real test - can the fine-tuned model actually use tools when orchestrated by CrewAI?

In [ ]:
# =============================================================================
# Setup CrewAI with tools (same as notebook 03)
# =============================================================================

from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool
from qdrant_client import QdrantClient
import os

# Connect to Qdrant
QDRANT_URL = os.environ.get('QDRANT_URL')
qdrant = QdrantClient(url=QDRANT_URL, port=443, https=True, verify=False)
collection_name = "research_papers"

# Verify Qdrant connection
try:
    collection_info = qdrant.get_collection(collection_name)
    print(f"Connected to Qdrant: {collection_info.points_count} vectors in '{collection_name}'")
except Exception as e:
    print(f"Qdrant not available: {e}")
    print("Run notebook 02 first to index papers.")

# Initialize embedding client for search tool
embedding_client = OpenAI(base_url=LITELLM_ENDPOINT, api_key=LITELLM_KEY)

# =============================================================================
# Define the same tools as notebook 03
# =============================================================================

@tool("Search Papers")
def search_papers(query: str) -> str:
    """
    Search the Qdrant vector database for papers matching a semantic query.
    Use this when you need to find papers about a specific topic or technique.
    """
    try:
        response = embedding_client.embeddings.create(model="nomic-embed", input=query)
        query_vector = response.data[0].embedding
        results = qdrant.search(collection_name=collection_name, query_vector=query_vector, limit=3)
        
        formatted = []
        for i, result in enumerate(results, 1):
            title = result.payload['metadata']['title']
            authors = result.payload['metadata']['authors']
            text = result.payload['text'][:500]
            formatted.append(f"[{i}] {title}\nAuthors: {authors}\nExcerpt: {text}...")
        
        return "\n\n".join(formatted) if formatted else "No relevant papers found."
    except Exception as e:
        return f"Search failed: {str(e)}"

@tool("Get Paper Details")
def get_paper_details(paper_title_fragment: str) -> str:
    """
    Get the full content of a specific paper by title.
    Use this when you need detailed information about a paper you've identified.
    """
    # Scroll through collection to find matching paper
    scroll_result = qdrant.scroll(collection_name=collection_name, limit=100, with_payload=True)
    
    for point in scroll_result[0]:
        title = point.payload['metadata']['title']
        if paper_title_fragment.lower() in title.lower():
            return f"Title: {title}\nAuthors: {point.payload['metadata']['authors']}\n\nContent:\n{point.payload['text']}"
    
    return f"No paper found matching '{paper_title_fragment}'"

print("CrewAI tools defined: search_papers, get_paper_details")

In [ ]:
def run_crewai_test(model_name: str, model_label: str) -> dict:
    """
    Run a single CrewAI agent with tools and capture whether it used them.
    
    Returns dict with: model, used_tool, tool_name, output_preview
    """
    print(f"\n{'='*70}")
    print(f"Testing {model_label}: {model_name}")
    print('='*70)
    
    # Create LLM instance for this model
    llm = LLM(
        model=model_name,
        base_url=f"{LITELLM_ENDPOINT}/v1",
        api_key=LITELLM_KEY,
        temperature=0.7,
        max_tokens=2000
    )
    
    # Create agent with tools
    researcher = Agent(
        role="Research Paper Analyst",
        goal="Search and analyze research papers using the available tools",
        backstory="You are a researcher with access to a paper database. You MUST use tools to find information.",
        tools=[search_papers, get_paper_details],
        verbose=True,
        llm=llm
    )
    
    # Create task that requires tool use
    task = Task(
        description="""Find papers about LoRA and parameter-efficient fine-tuning.

YOU MUST USE YOUR TOOLS:
1. Use 'Search Papers' to find relevant papers
2. Use 'Get Paper Details' to get more information on interesting papers
3. Summarize what you found

DO NOT make up information. Only report what you find using tools.""",
        expected_output="A summary of papers found using the search tools, with actual paper titles and authors.",
        agent=researcher
    )
    
    # Create and run crew
    crew = Crew(
        agents=[researcher],
        tasks=[task],
        process=Process.sequential,
        verbose=True
    )
    
    try:
        result = crew.kickoff()
        output = str(result)
        
        # Check if output contains evidence of tool use
        # (real paper titles from Qdrant, not generic text)
        tool_indicators = ["FouRA", "LoRA-FA", "Flat-LoRA", "ABM-LoRA", "RaSA", "Authors:"]
        used_tool = any(indicator in output for indicator in tool_indicators)
        
        return {
            "model": model_name,
            "label": model_label,
            "used_tool": used_tool,
            "output_preview": output[:500],
            "full_output": output,
            "error": None
        }
    except Exception as e:
        return {
            "model": model_name,
            "label": model_label,
            "used_tool": False,
            "output_preview": None,
            "full_output": None,
            "error": str(e)
        }

print("CrewAI test function defined.")

In [ ]:
# =============================================================================
# Run CrewAI comparison: Base vs Fine-Tuned
# =============================================================================
# This is the real test - does the fine-tuned model actually use tools in CrewAI?

import os
os.environ['CREWAI_TELEMETRY'] = 'false'

print("Running CrewAI comparison...")
print("This will run the same agent/task with both models.\n")

# Test base model
base_crewai = run_crewai_test(BASE_MODEL, "Base Model")

# Test fine-tuned model (if available)
if finetuned_available:
    finetuned_crewai = run_crewai_test(FINETUNED_MODEL, "Fine-Tuned Model")
else:
    print(f"\nSkipping fine-tuned model test - {FINETUNED_MODEL} not available")
    finetuned_crewai = None

In [ ]:
# =============================================================================
# CrewAI Results Summary
# =============================================================================

print("\n" + "="*70)
print("CREWAI COMPARISON RESULTS")
print("="*70)

print(f"\nBASE MODEL ({BASE_MODEL}):")
print(f"  Used tools: {'YES' if base_crewai['used_tool'] else 'NO'}")
if base_crewai['error']:
    print(f"  Error: {base_crewai['error']}")
else:
    print(f"  Output preview: {base_crewai['output_preview'][:200]}...")

if finetuned_crewai:
    print(f"\nFINE-TUNED MODEL ({FINETUNED_MODEL}):")
    print(f"  Used tools: {'YES' if finetuned_crewai['used_tool'] else 'NO'}")
    if finetuned_crewai['error']:
        print(f"  Error: {finetuned_crewai['error']}")
    else:
        print(f"  Output preview: {finetuned_crewai['output_preview'][:200]}...")
    
    # Summary
    print("\n" + "-"*70)
    if finetuned_crewai['used_tool'] and not base_crewai['used_tool']:
        print("CONCLUSION: Fine-tuning IMPROVED tool use in CrewAI context")
    elif finetuned_crewai['used_tool'] and base_crewai['used_tool']:
        print("CONCLUSION: Both models use tools (fine-tuning may still improve quality)")
    elif not finetuned_crewai['used_tool'] and not base_crewai['used_tool']:
        print("CONCLUSION: Neither model used tools - may need more training data")
    else:
        print("CONCLUSION: Unexpected result - base model used tools but fine-tuned didn't")